In [5]:
import numpy as np
import pandas as pd
from typing import List, Dict
import nltk
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
from collections import defaultdict
import hashlib
import json

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

# Explicitly check and download 'punkt_tab' as it's required by sent_tokenize
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')


class HierarchicalRAG:

    def __init__(self, chunk_size=5):
        self.documents = {}
        self.doc_embeddings = {}
        self.section_embeddings = {}
        self.sentence_embeddings = {}
        self.hierarchy = defaultdict(lambda: defaultdict(list))
        self.vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
        self.chunk_size = chunk_size
        self.is_fitted = False

    def add_document(self, doc_id: str, content: str, metadata: Dict = None):
        self.documents[doc_id] = {
            'content': content,
            'metadata': metadata or {},
            'sections': self._split_into_sections(content),
            'sentences': sent_tokenize(content)
        }
        self._build_hierarchy(doc_id)

    def _split_into_sections(self, content: str) -> List[str]:
        sections = [p.strip() for p in content.split('\n\n') if p.strip()]
        if not sections:
            words = content.split()
            sections = [
                ' '.join(words[i:i+self.chunk_size*10])
                for i in range(0, len(words), self.chunk_size*10)
            ]
        return sections

    def _build_hierarchy(self, doc_id: str):
        doc = self.documents[doc_id]
        for i, section in enumerate(doc['sections']):
            section_id = f"{doc_id}_sec_{i}"
            self.hierarchy[doc_id]['sections'].append(section_id)

            section_sentences = sent_tokenize(section)
            for j, sentence in enumerate(section_sentences):
                sentence_id = f"{section_id}_sent_{j}"
                self.hierarchy[doc_id]['sentences'].append(sentence_id)
                self.hierarchy[section_id]['sentences'].append(sentence_id)

    def fit_embeddings(self):
        all_texts = []
        for doc in self.documents.values():
            all_texts.append(doc['content'])
            all_texts.extend(doc['sections'])
            all_texts.extend(doc['sentences'])

        self.vectorizer.fit(all_texts)
        self.is_fitted = True

        for doc_id, doc in self.documents.items():
            self.doc_embeddings[doc_id] = self._get_embedding(doc['content'])

            for i, section in enumerate(doc['sections']):
                section_id = f"{doc_id}_sec_{i}"
                self.section_embeddings[section_id] = self._get_embedding(section)

            for i, sentence in enumerate(doc['sentences']):
                sentence_id = f"{doc_id}_sent_{i}"
                self.sentence_embeddings[sentence_id] = self._get_embedding(sentence)

    def _get_embedding(self, text: str):
        if not self.is_fitted:
            raise ValueError("Call fit_embeddings() first")
        return self.vectorizer.transform([text]).toarray()[0]

    def retrieve(self, query: str, top_k: int = 5):
        query_emb = self._get_embedding(query)
        similarities = []

        for sent_id, sent_emb in self.sentence_embeddings.items():
            sim = cosine_similarity([query_emb], [sent_emb])[0][0]
            similarities.append((sent_id, sim))

        similarities.sort(key=lambda x: x[1], reverse=True)

        results = []
        for sent_id, score in similarities[:top_k]:
            results.append({
                'id': sent_id,
                'text': self._get_text_by_id(sent_id),
                'score': float(score),
                'context': self._get_context(sent_id)
            })

        return results

    def _get_text_by_id(self, text_id: str):
        parts = text_id.split('_')
        doc_id = parts[0]
        sent_idx = int(parts[-1])
        return self.documents[doc_id]['sentences'][sent_idx]

    def _get_context(self, sent_id: str, window: int = 2):
        parts = sent_id.split('_')
        doc_id = parts[0]
        sent_idx = int(parts[-1])
        sentences = self.documents[doc_id]['sentences']
        start = max(0, sent_idx - window)
        end = min(len(sentences), sent_idx + window + 1)
        return ' '.join(sentences[start:end])

    def generate_answer(self, query: str, retrieved_items: List[Dict]):
        if not retrieved_items:
            return "No relevant information found."

        contexts = []
        for item in retrieved_items[:3]:
            contexts.append(f"- {item['text']}")

        answer = "Based on the retrieved information:\n"
        answer += '\n'.join(contexts)
        answer += f"\n\nThis addresses the query: '{query}'"
        return answer


def create_sample_documents():
    return {
        'doc1': """
        Artificial Intelligence (AI) is transforming the world.
        Machine learning enables computers to learn from data.
        Deep learning uses neural networks with multiple layers.
        """,
        'doc2': """
        Climate change is a global challenge.
        Renewable energy like solar and wind reduces carbon emissions.
        Individual actions help protect the environment.
        """,
        'doc3': """
        The human brain contains billions of neurons.
        Memory formation strengthens neural connections.
        Sleep plays a crucial role in memory consolidation.
        """
    }


if __name__ == "__main__":

    rag = HierarchicalRAG(chunk_size=3)

    docs = create_sample_documents()
    for doc_id, content in docs.items():
        rag.add_document(doc_id, content)

    rag.fit_embeddings()

    query = "How does machine learning work?"
    results = rag.retrieve(query, top_k=3)

    for i, result in enumerate(results, 1):
        print(f"\n{i}. Score: {result['score']:.3f}")
        print("Text:", result['text'])
        print("Context:", result['context'])

    print("\nGenerated Answer:\n")
    print(rag.generate_answer(query, results))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.



1. Score: 0.559
Text: Machine learning enables computers to learn from data.
Context: 
        Artificial Intelligence (AI) is transforming the world. Machine learning enables computers to learn from data. Deep learning uses neural networks with multiple layers.

2. Score: 0.241
Text: Deep learning uses neural networks with multiple layers.
Context: 
        Artificial Intelligence (AI) is transforming the world. Machine learning enables computers to learn from data. Deep learning uses neural networks with multiple layers.

3. Score: 0.000
Text: 
        Artificial Intelligence (AI) is transforming the world.
Context: 
        Artificial Intelligence (AI) is transforming the world. Machine learning enables computers to learn from data. Deep learning uses neural networks with multiple layers.

Generated Answer:

Based on the retrieved information:
- Machine learning enables computers to learn from data.
- Deep learning uses neural networks with multiple layers.
- 
        Artificial In